In [1]:
# import necessary packages
import folium
from pathlib import Path
import pandas as pd
import geopandas as gpd
import platform

In [2]:
# read in the wetlands datasets from previous processing work in wetland_explore

# establish paths to datasets
project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'/ 'wetland_explore'
figure_fldr = project_fldr/"figures"

# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
# uploading wetlands that are bigger than a hectare
html_map_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'html_maps'

wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

simpler_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'simpler_wetlands'


Box folder found at /Users/masonbunker/Library/CloudStorage/Box-Box
True


In [3]:
# read in wetlands file
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")


In [4]:
# make new dfsfor mapping
wetlands_map = wetlands.copy()

# make geometries valid
wetlands_map["geometry"] = wetlands_map.geometry.make_valid()

cols = ['C System', "C Subsystem", "C Code", "C Class", 'Hectares', 'geometry']

wetlands_map = wetlands_map[cols]


# new datasets to work with
riverine = wetlands_map.loc[ lambda df: df["C System"] == "Riverine"]
palustrine = wetlands_map.loc[ lambda df: df["C System"] == "Palustrine"]

In [5]:
# put into the correct crs
# first save to appropriate crs
palustrine= palustrine.to_crs("EPSG:4326")
riverine = riverine.to_crs("EPSG:4326")


In [6]:
# define functions to perform actions
def simpler(df, name, tolerance):
    df_simpler = df.copy()
    df_simpler["geometry"] = df.geometry.simplify(
        tolerance=tolerance,
        preserve_topology=True
        )
    return df_simpler


#chatgpt helped to finetune this code
def vertices_before(df):
    # count vertices 
    def count_vertices(df):
        if df.geom_type == "Polygon":
            return len(df.exterior.coords) + sum(
                len(ring.coords) for ring in df.interiors
            )

        elif df.geom_type == "MultiPolygon":
            return sum(count_vertices(poly) for poly in df.geoms)

        return 0
    return print(f"vertices before: {df.geometry.apply(count_vertices).sum():,}") # purple at end specifies the commas

def vertices_after(df):
        # count vertices after
    def count_vertices(df):
        if df.geom_type == "Polygon":
            return len(df.exterior.coords) + sum(
                len(ring.coords) for ring in df.interiors
            )

        elif df.geom_type == "MultiPolygon":
            return sum(count_vertices(poly) for poly in df.geoms)

        return 0
    return print(f"vertices after: {df.geometry.apply(count_vertices).sum():,}") # purple at end specifies the commas

~~Goal is to simplify below 1,000,000 to start!~~

# Need to get below 5mb as ideal, below 10 is acceptable

In [7]:
# palustrine simpler test
print("PALUSTRINE")
vertices_before(palustrine)
palustrine_simpler = simpler(palustrine, "palustrine", 500)
vertices_after(palustrine_simpler)

#size check
size_mb = len(palustrine_simpler.to_json().encode("utf-8")) / (1024 ** 2)
print(f"Palustrine ({palustrine_simpler.shape[0]:,} rows): {size_mb:.2f} MB")

PALUSTRINE
vertices before: 29,972,411
vertices after: 616,209
Palustrine (96,236 rows): 45.72 MB


In [8]:
for tolerance in [100, 150, 200, 300, 500, 750, 1000]:
    test = simpler(palustrine, "palustrine", tolerance)

    size_mb = len(test.to_json().encode("utf-8")) / (1024 ** 2)

    print(
        f"Tolerance: {tolerance:>4} | "
        f"Vertices: {test.geometry.apply(
            lambda geom: len(geom.exterior.coords)
            if geom.geom_type == "Polygon"
            else sum(len(poly.exterior.coords) for poly in geom.geoms)
            if geom.geom_type == "MultiPolygon"
            else 0
        ).sum():,} | "
        f"Size: {size_mb:.4f} MB"
    )

Tolerance:  100 | Vertices: 529,350 | Size: 45.7248 MB
Tolerance:  150 | Vertices: 529,350 | Size: 45.7248 MB
Tolerance:  200 | Vertices: 529,350 | Size: 45.7248 MB
Tolerance:  300 | Vertices: 529,350 | Size: 45.7248 MB
Tolerance:  500 | Vertices: 529,350 | Size: 45.7248 MB
Tolerance:  750 | Vertices: 529,350 | Size: 45.7248 MB
Tolerance: 1000 | Vertices: 529,350 | Size: 45.7248 MB


In [ ]:
# riverine simpler
print("RIVERINE")
vertices_before(riverine)
riverine_simpler = simpler(riverine, "riverine", 150)
vertices_after(riverine_simpler)

#size check
size_mb = len(riverine_simpler.to_json().encode("utf-8")) / (1024 ** 2)
print(f"Riverine ({riverine_simpler.shape[0]:,} rows): {size_mb:.2f} MB")

RIVERINE
vertices before: 136,039,508
vertices after: 3,598,284


In [ ]:
# palustrine_simpler.explore()

In [ ]:
# riverine_simpler.explore()

In [ ]:
# save the simplified files to call more easily

palustrine_simpler.to_file(
    simpler_path/"palustrine_simpler.geojson", driver= "GeoJSON")
riverine_simpler.to_file(
    simpler_path/"riverine_simpler.geojson", driver= "GeoJSON")